In [ ]:
import h5py
import numpy as np


def load_pixel_dataset(dataset_path, compact_dataset=False):
    with h5py.File(dataset_path, "r") as f:
        observations = {
            "rgb": f["observations/rgb"][...],
            "depth": f["observations/depth"][...],
            "intrinsics": f["observations/intrinsics"][...],
            "extrinsics": f["observations/extrinsics"][...],
        }
        actions = f["actions"][...].astype(np.float32)
        terminals = f["terminals"][...].astype(np.float32)
        qpos = f["qpos"][...] if "qpos" in f else None
        qvel = f["qvel"][...] if "qvel" in f else None

    if compact_dataset:
        valids = 1.0 - terminals
        new_terminals = np.concatenate([terminals[1:], [1.0]])
        terminals = np.minimum(terminals + new_terminals, 1.0).astype(np.float32)
        return dict(
            observations=observations,
            actions=actions,
            terminals=terminals,
            valids=valids,
            qpos=qpos,
            qvel=qvel,
        )
    else:
        ob_mask = (1.0 - terminals).astype(bool)
        next_ob_mask = np.concatenate([[False], ob_mask[:-1]])
        next_observations = {k: v[next_ob_mask] for k, v in observations.items()}
        observations = {k: v[ob_mask] for k, v in observations.items()}
        actions = actions[ob_mask]
        new_terminals = np.concatenate([terminals[1:], [1.0]])
        terminals = new_terminals[ob_mask].astype(np.float32)
        return dict(
            observations=observations,
            next_observations=next_observations,
            actions=actions,
            terminals=terminals,
            qpos=qpos,
            qvel=qvel,
        )